# vit_small × {FunnyBirds, dSprites, ColoredMNIST} — XAI baselines

Companion to `dinov3_unfolded/walkthrough.ipynb`. Two parts:

1. **Train a small ViT** — the project's finetune CLI applied to
   the 22 M-parameter `vit_small_patch16_224.augreg_in21k_ft_in1k`
   on each of three datasets. Each dataset gets one short cell
   that shells out to `uv run python -m experiments.train_probe`.
   Skip a cell if the checkpoint already exists.
2. **Inspect with two published baselines** — once a checkpoint
   is trained, point the LeGrad and Chefer cells at it and look
   at heatmaps. Pick the layer / block of interest at the top of
   each cell. The point is to compare these well-known methods
   against our AttnLRP/CRP attributions on a model whose ground
   truth (what the model recognises) we control via the choice
   of training set.

Why these three datasets:

| Dataset | What you control | XAI use |
|---|---|---|
| **FunnyBirds**   | per-part ground truth | does the heatmap localise the right bird parts? |
| **dSprites**     | shape / scale / position factors | does the heatmap track the shape pixels? |
| **ColoredMNIST** | colour↔digit correlation, broken at test time | does the model learn shape (test ≥ 96 %) or shortcut on colour? |

## 1. Setup

In [ ]:
%cd ../../..
%ls
from __future__ import annotations
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from experiments.datasets import load as load_dataset
from experiments.models import BASES, build_probe
from experiments.viz_unfolded import to_display

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUNS_DIR = REPO_ROOT / 'data' / 'runs'
print(f'device   : {DEVICE}')
print(f'runs dir : {RUNS_DIR}')

## 2. Train vit_small on each dataset

Three short cells, one per dataset. Each spells out the recipe in
a comment and shells out to the project's `finetune` CLI with
`--from-scratch`. All three write a timestamped run directory under
`data/runs/finetune_vit_small_<dataset>/<UTC ts>/` containing:

* `best.pt` — best-by-val_acc backbone + head + metadata
* `config.json` — full Typer params (reproducibility)
* `metrics.csv` — per-epoch logs (Lightning CSVLogger)

Hyperparams are tunable; if a run misses the > 96 % accuracy
target, raise `--llrd`, lengthen `--warmup-epochs`, or grow
`--epochs`. The defaults below match the recipe in the project
plan.

Skip any cell whose checkpoint already exists (find with
`ls data/runs/finetune_vit_small_*/`).

### 2a. FunnyBirds (50-class part-based bird classification, ~6 h on a single GPU)

RandAugment + Mixup + LLRD + cosine schedule + label smoothing —
the standard ImageNet ViT fine-tune recipe. `clean_only=True`
filters the ~41 % of train samples with part ablations.

In [ ]:
!uv run python -m experiments.train_probe finetune \
    --from-scratch \
    --base vit_small \
    --dataset funny_birds \
    --head linear \
    --epochs 50 \
    --patience 10 \
    --backbone-lr 1e-5 \
    --head-lr 1e-3 \
    --weight-decay 0.05 \
    --batch-size 64 \
    --accumulate-grad-batches 2 \
    --llrd 0.65 \
    --scheduler cosine \
    --warmup-epochs 5 \
    --randaugment \
    --mixup 0.8 \
    --label-smoothing 0.1 \
    --num-workers 4

### 2b. dSprites — 3-class shape (square / ellipse / heart, ~30 min)

Synthetic, no augmentation needed. The 3-class shape task is
essentially trivial for a vit_small from in21k init.

In [ ]:
!uv run python -m experiments.train_probe finetune \
    --from-scratch \
    --base vit_small \
    --dataset dsprites \
    --head linear \
    --epochs 10 \
    --patience 3 \
    --backbone-lr 1e-5 \
    --head-lr 1e-4 \
    --batch-size 128 \
    --accumulate-grad-batches 1 \
    --no-augment \
    --scheduler cosine \
    --warmup-epochs 2 \
    --num-workers 4

### 2c. ColoredMNIST — 10-class digit, colour↔digit 99 % correlation (~1 h)

Geometric augmentation only — **no** ColorJitter (colour is signal
not noise). Test split has uniformly random colours: > 96 % means
the model learned digit shape; ~10 % means the colour shortcut
won. That gap is the whole point of the dataset.

In [ ]:
!uv run python -m experiments.train_probe finetune \
    --from-scratch \
    --base vit_small \
    --dataset colored_mnist \
    --head linear \
    --epochs 20 \
    --patience 5 \
    --backbone-lr 1e-5 \
    --head-lr 3e-4 \
    --batch-size 128 \
    --accumulate-grad-batches 1 \
    --augment \
    --label-smoothing 0.1 \
    --scheduler cosine \
    --warmup-epochs 2 \
    --num-workers 4

## 3. Load a trained checkpoint

Set `CKPT` to the run you want to inspect. The default below
globs the most recent FunnyBirds run; change the dataset name in
the glob to switch.

In [ ]:
import json, glob

# Pick a dataset and the most recent run for it.
DATASET = 'funny_birds'   # 'funny_birds' | 'dsprites' | 'colored_mnist'
_runs = sorted(glob.glob(str(RUNS_DIR / f'finetune_vit_small_{DATASET}' / '*' / 'best.pt')))
if not _runs:
    raise FileNotFoundError(
        f'No checkpoint found under {RUNS_DIR}/finetune_vit_small_{DATASET}/. '
        f'Run section 2{["a","b","c"][["funny_birds","dsprites","colored_mnist"].index(DATASET)]} first.'
    )
CKPT = Path(_runs[-1])
print(f'checkpoint: {CKPT}')
print(f'config    : {(CKPT.parent / "config.json").read_text()[:300]} ...')

In [ ]:
ckpt = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = build_probe(
    base=ckpt['base'], head=ckpt['head'],
    num_classes=ckpt['num_classes'],
    head_kwargs=ckpt.get('head_kwargs', {}),
).to(DEVICE)
model.backbone.load_state_dict(ckpt['backbone_state_dict'])
model.head.load_state_dict(ckpt['head_state_dict'])
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print(f'val_acc  : {ckpt["val_acc"]:.4f}')
print(f'val_loss : {ckpt["val_loss"]:.4f}')
print(f'embed_dim: {model.backbone.embed_dim}, depth: {len(model.backbone.blocks)}, heads: {model.backbone.blocks[0].attn.num_heads}')

# Per-batch normalize closure (datasets emit unnormalized [0,1]).
from experiments.models import build_base
_base = build_base(ckpt['base'])
normalize = _base.get_normalize()
transform = _base.get_transform()

In [ ]:
# Pick a focal image: first correctly-classified test sample.
if DATASET == 'funny_birds':
    ds = load_dataset('funny_birds', transform=transform, split='test')
elif DATASET == 'colored_mnist':
    ds = load_dataset('colored_mnist', transform=transform, split='test')
else:
    ds = load_dataset(DATASET, transform=transform)

focal_image = focal_class = focal_index = None
for i in range(0, len(ds), max(1, len(ds) // 50)):
    x_, y_ = ds[i]
    x_dev = x_.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(normalize(x_dev)).argmax(-1).item()
    if pred == int(y_):
        focal_image = x_dev
        focal_class = pred
        focal_index = i
        break
if focal_image is None:
    raise RuntimeError('no correctly-classified sample found in first 50 strided samples')

print(f'focal: index {focal_index}, class {focal_class}')
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(to_display(focal_image)); ax.axis('off')
ax.set_title(f'class {focal_class}')
plt.show()

## 4a. LeGrad — feature formation sensitivity

Bousselham et al., arXiv 2404.03214. Computes a heatmap by
differentiating the prediction w.r.t. the *attention weights* of
a chosen set of blocks, then aggregating per-token sensitivities
back to image space. Designed for ViTs. Library:
[`legrad_torch`](https://github.com/WalBouss/LeGrad).

The library's public wrapper expects an OpenCLIP `model.visual`
interface, so we hook directly via the lower-level method: the
library only needs to know which attention modules to attach to
and how to pull out the model logits. We pass our timm
`model.backbone.blocks[*].attn` list and the head as the logit
source.

**TARGET_BLOCKS** at the top of the cell controls which blocks'
attention weights contribute to the heatmap — change to inspect
shallow vs deep evidence. The default (all blocks) matches the
paper's recommended setting for ViT classification.

In [ ]:
# === LeGrad — pick which blocks contribute ===
TARGET_BLOCKS = list(range(len(model.backbone.blocks)))

# LeGrad's implementation hooks attn.softmax outputs via a
# `register_attention_hook(model)` helper. We do the same thing
# inline (the upstream helper is ~30 lines and tightly coupled to
# OpenCLIP modules; reproducing it on our timm Attention is short
# and avoids the OpenCLIP-shim headache).

_attn_maps = []   # one per block in forward order
_handles = []
def _capture_attn(_mod, _inp, _out):
    _attn_maps.append(_out)   # (B, num_heads, N, N), softmax output
for i in TARGET_BLOCKS:
    _handles.append(
        model.backbone.blocks[i].attn.softmax.register_forward_hook(_capture_attn)
    )

try:
    # Forward + backward through one-hot logit. Switch the backbone
    # back to no_grad-eligible state but turn on autograd just for
    # this one image.
    for p in model.parameters():
        p.requires_grad_(False)
    x = normalize(focal_image).detach().clone().requires_grad_(False)
    _attn_maps.clear()
    # Each captured attention map needs grad → make them retain grad.
    # Trick: do a fresh forward, then call .retain_grad() on each.
    logits = model(x)
    for a in _attn_maps:
        a.retain_grad()
    logits[0, focal_class].backward()

    # LeGrad heatmap: per-block, take grad of the attn map, average
    # over heads, sum out source token (q-axis), keep cls-row, ReLU,
    # then aggregate across blocks (mean). The patch-token portion
    # of cls-row reshapes to a square spatial map.
    npt = int(getattr(model, 'num_prefix_tokens', 1))
    per_block = []
    for a in _attn_maps:
        if a.grad is None:
            continue
        g = a.grad                          # (1, H, N, N)
        per = (g * a).clamp_min(0).mean(dim=1)   # (1, N, N) — head-mean
        # Cls-row, drop prefix tokens, keep spatial:
        cls_row = per[0, 0, npt:]                # (P,)
        per_block.append(cls_row)
    if not per_block:
        raise RuntimeError('no gradients captured — ensure focal_image was the model input')
    legrad_score = torch.stack(per_block).mean(dim=0)     # (P,)
    side = int(legrad_score.numel() ** 0.5)
    heatmap = legrad_score.reshape(side, side).detach().cpu().numpy()
finally:
    for h in _handles:
        h.remove()

fig, (ax_img, ax_hm) = plt.subplots(1, 2, figsize=(6, 3))
ax_img.imshow(to_display(focal_image)); ax_img.axis('off')
ax_img.set_title(f'class {focal_class}', fontsize=9)
vmax = abs(heatmap).max() or 1.0
ax_hm.imshow(heatmap, cmap='seismic', vmin=-vmax, vmax=vmax)
ax_hm.axis('off'); ax_hm.set_title(f'LeGrad — blocks {TARGET_BLOCKS}', fontsize=9)
plt.tight_layout(); plt.show()

## 4b. Chefer's method — gradient-weighted attention rollout

Chefer, Gur, Wolf. CVPR 2021, arXiv 2012.09838. Per block:
compute relevance from `(grad ⊙ A).clamp(0)` averaged over heads,
then roll up across blocks via $R \leftarrow R + R_{block} \cdot R$
starting from $R = I$. Code reference:
[`hila-chefer/Transformer-Explainability`](https://github.com/hila-chefer/Transformer-Explainability)
(`baselines/ViT/ViT_explanation_generator.py`). Vendored inline
because the upstream package isn't on PyPI and the rollout is short.

**UP_TO_BLOCK** at the top of the cell controls how deep the
rollup goes — set to a smaller integer to inspect the relevance
as it accumulates through the stack. Default = all blocks.

In [ ]:
# === Chefer's rollout — pick how many blocks to roll up ===
UP_TO_BLOCK = len(model.backbone.blocks) - 1   # 0-indexed inclusive

_attn_maps = []
_handles = []
def _capture_attn(_mod, _inp, _out):
    _attn_maps.append(_out)
for i in range(UP_TO_BLOCK + 1):
    _handles.append(
        model.backbone.blocks[i].attn.softmax.register_forward_hook(_capture_attn)
    )

try:
    x = normalize(focal_image).detach().clone()
    _attn_maps.clear()
    logits = model(x)
    for a in _attn_maps:
        a.retain_grad()
    logits[0, focal_class].backward()

    # Build per-block relevance R_block = E_h[(grad ⊙ A)+]
    blocks_R = []
    for a in _attn_maps:
        if a.grad is None:
            continue
        # Per Chefer Eq. 5: average over heads, ReLU, ignore neg.
        Rb = (a.grad * a).clamp_min(0).mean(dim=1)[0]   # (N, N)
        blocks_R.append(Rb)
    if not blocks_R:
        raise RuntimeError('no gradients captured')

    # Roll up: R = I; for each block, R = R + Rb @ R (Chefer Eq. 6).
    N = blocks_R[0].size(0)
    R = torch.eye(N, device=blocks_R[0].device)
    for Rb in blocks_R:
        R = R + Rb @ R

    # Pull cls-token row, drop prefix tokens, reshape to spatial.
    npt = int(getattr(model, 'num_prefix_tokens', 1))
    cls_row = R[0, npt:]
    side = int(cls_row.numel() ** 0.5)
    heatmap = cls_row.reshape(side, side).detach().cpu().numpy()
finally:
    for h in _handles:
        h.remove()

fig, (ax_img, ax_hm) = plt.subplots(1, 2, figsize=(6, 3))
ax_img.imshow(to_display(focal_image)); ax_img.axis('off')
ax_img.set_title(f'class {focal_class}', fontsize=9)
vmax = abs(heatmap).max() or 1.0
ax_hm.imshow(heatmap, cmap='seismic', vmin=-vmax, vmax=vmax)
ax_hm.axis('off')
ax_hm.set_title(f'Chefer rollout — blocks 0..{UP_TO_BLOCK}', fontsize=9)
plt.tight_layout(); plt.show()

## 5. Notes

* **Attention site.** Both methods hook `attn.softmax`, which is
  the post-softmax attention weight tensor `(B, H, N, N)` in
  both timm `Attention` and our `TimmAttentionUnfolded`. No
  composite needed — these are stock-model methods.
* **Why no statistics.** This notebook is for *manual* sanity
  checks: "does the heatmap look reasonable for this image?".
  Quantitative comparison (faithfulness, robustness, etc.) is a
  separate exercise; see Quantus / SaCo for harnesses.
* **Comparing to our AttnLRP/CRP.** To run our own attribution
  on the same focal image, build `AttnLRPCombinedComposite` +
  `CondAttribution` as in the DINOv3 walkthrough — the API is
  identical, only the model differs.
* **ColoredMNIST sanity.** If `val_acc` on the test split is
  near 10 %, the model learned the colour shortcut and broke at
  test time (where the colour↔digit correlation is removed). A
  colour-shortcut model's heatmap will fire on the *whole* digit
  silhouette regardless of position; a shape-aware model's
  heatmap should track stroke geometry. That contrast is what
  makes ColoredMNIST useful for evaluating XAI methods.